In [ ]:
%cd /workspaces/Pointcept
from pointcept.engines.defaults import  default_setup
from pointcept.engines.defaults import (
    default_argument_parser,
    default_config_parser,
)
from pointcept.engines.train import TRAINERS
import torch
import numpy as np

In [ ]:
args = default_argument_parser().parse_args(
"""
--config-file /workspaces/Pointcept/configs/scannet/pipeline-sculpting-ptv3.py 
--num-gpus 0 
--options num_workers=0 num_worker_per_gpu=0 batch_size=1 
          weight=/workspaces/Pointcept/artifacts/spu8by4j-model_last:v80""".split()
)
cfg = default_config_parser(args.config_file, args.options)

In [ ]:
cfg = default_setup(cfg)
#cfg.num_worker_per_gpu=0
trainer = TRAINERS.build(dict(type=cfg.train.type, cfg=cfg))

In [ ]:
train_loader = trainer.train_loader
i, b = next(enumerate(train_loader))

b

In [ ]:
b.keys()

In [ ]:
from pointcept.utils.events import EventStorage, ExceptionWriter
import pointcept.utils.comm as comm

with EventStorage() as trainer.storage, ExceptionWriter():
    # => before train
    trainer.before_train()
    trainer.logger.info(">>>>>>>>>>>>>>>> Start Training >>>>>>>>>>>>>>>>")
    for trainer.epoch in range(100):
        # => before epoch
        trainer.model.train()
        trainer.data_iterator = enumerate(trainer.train_loader)
        trainer.before_epoch()
        # => run_epoch
        for (
            trainer.comm_info["iter"],
            trainer.comm_info["input_dict"],
        ) in trainer.data_iterator:
            # => before_step
            trainer.before_step()
            # => run_step
            trainer.run_step()
            # => after_step
            trainer.after_step()
        # => after epoch
        trainer.after_epoch()
    # => after train
    trainer.after_train()

In [ ]:
trainer.comm_info["model_output_dict"]